# Student Course Enrollments — Dataset Documentation

**Dataset:** `course_enrollments.csv`

**Purpose:** Tracks student course enrollments, academic performance, course completion, and credits attempted and earned within the synthetic Graduation Intelligence student information system. Supports graduation credit calculations, identification of failed courses, and monitoring of academic progress toward graduation.

**Record level:** One row per student-course enrollment.

**Primary key:** `enrollment_id`

**Foreign keys:**
- `student_id` → `students.csv`
- `course_id` → `course_catalogue.csv`

**Fields:**

- `enrollment_id`: Unique enrollment identifier using the format `ENR00001`.
- `student_id`: Student identifier linking enrollment records to the student dataset.
- `course_id`: Course identifier linking enrollment records to the course catalogue.
- `school_year`: Academic year associated with the enrollment, formatted as `YYYY-YY` (e.g., `2025-26`).
- `credits_attempted`: Credit value of the enrolled course, inherited from the course catalogue.
- `enrollment_status`: Indicates whether the enrollment is Completed or In Progress.
- `final_grade`: Numeric final course grade ranging from 0 to 100. Null for courses currently in progress.
- `course_outcome`: Academic outcome of the enrollment (Passed, Failed, or In Progress).
- `credits_earned`: Number of credits earned through successful course completion.
- `attempted_number`: Course attempt number, initialized to 1 for all enrollments.

**Business rules:**

- Each student receives a randomly generated number of course enrollments based on grade level:
  - Grade 9: 6–8 courses.
  - Grade 10: 12–16 courses.
  - Grade 11: 18–24 courses.
  - Grade 12: 24–30 courses.
- Courses are randomly selected from the course catalogue without replacement for each student.
- Course enrollment start years are randomly generated based on the student's current grade level.
- Courses beginning in 2026 receive an enrollment status of In Progress.
- Courses beginning before 2026 receive an enrollment status of Completed.
- Completed courses receive a randomly generated final grade using a normal distribution with a mean of 78 and standard deviation of 14, rounded to the nearest whole number and restricted to 0–100.
- In-progress courses have null final grades.
- A final grade of 60 or higher results in a Passed outcome.
- A final grade below 60 results in a Failed outcome.
- In-progress courses receive an In Progress outcome.
- Credits attempted are determined by the credit value assigned in the course catalogue.
- Passed courses award the full attempted credit value.
- Failed and in-progress courses award 0 earned credits.
- Credits earned cannot exceed credits attempted.
- All course attempt numbers are initialized to 1.
- Enrollment IDs must be unique.
- Every `student_id` must exist in the student dataset.
- Every `course_id` must exist in the course catalogue.
- Random generation uses a fixed seed of 42 to support reproducibility.

**Data validation:**

- Verify all enrollment IDs are unique.
- Verify all student IDs exist in the student dataset.
- Verify all course IDs exist in the course catalogue.
- Verify credits earned never exceed credits attempted.
- Verify enrollments without a Passed outcome have zero earned credits.

**Data sources:**
- `students.csv`: Provides student identifiers and grade levels.
- `course_catalogue.csv`: Provides available course identifiers and credit values.

**Generation method:** Synthetic course enrollment records generated using Python, pandas, and NumPy random sampling.

**Limitations:** Course enrollments, academic years, grades, and outcomes are synthetically generated and do not represent actual student academic records. Course selection does not enforce prerequisite requirements, subject-specific graduation requirements, or realistic course sequencing. Enrollment counts represent total generated course records rather than verified historical transcripts. Course attempts are initialized to 1, and the dataset does not currently model repeated attempts or credit recovery. Credit awards follow project-defined passing criteria rather than institution-specific grading policies.

In [1]:
# Import libraries

import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive
from google.colab import files


In [2]:
# Connect to Google Drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Set random seed

rng = np.random.default_rng(42)

In [4]:
# Load Student Data and Course Catalogue

students = pd.read_csv('/content/drive/MyDrive/Graduation_Tracker/data/ground_truth/students.csv')
courses = pd.read_csv('/content/drive/MyDrive/Graduation_Tracker/data/ground_truth/course_catalogue.csv')


In [5]:
# Generate course attempts

records = []

counts = {9:(6,8), 10:(12, 16), 11:(18,24), 12:(24,30)}

for s in students.itertuples(index = False):
  low, high = counts[int(s.grade_level)]
  n = int(rng.integers(low,high+1))
  selected = rng.choice(courses.course_id.to_numpy(), size = n, replace = False)
  for cid in selected:
    start = int(rng.integers(2026-(int(s.grade_level) - 9), 2027))
    records.append((s.student_id, cid, start))

In [6]:
# Create enrollments DataFrame

enrollments = pd.DataFrame(records, columns = ['student_id', 'course_id', 'year_start'])

# Create enrollment id column
enrollments.insert(0, 'enrollment_id', [f'ENR{i:05d}' for i in range(1, len(enrollments) + 1)])

In [7]:
# Generate school_year

enrollments['school_year'] = enrollments.year_start.astype(str) + '-' + (enrollments.year_start + 1).astype(str).str[-2:]

In [8]:
# Create credits attempted column
enrollments = enrollments.merge(courses[['course_id', 'credit']], on = 'course_id', validate = 'many_to_one')

enrollments.rename(columns = {'credit':'credits_attempted'}, inplace = True)

In [9]:
# Create enrollment status column

enrollments['enrollment_status'] = np.where(enrollments.year_start == 2026, 'In Progress', 'Completed')

In [10]:
# Create final grade column

enrollments['final_grade'] = np.nan

mask = enrollments.enrollment_status.eq('Completed')

enrollments.loc[mask, 'final_grade'] = np.clip(rng.normal(78, 14, mask.sum()), 0, 100).round()

In [11]:
# Create course outcome column

enrollments['course_outcome'] = np.select([~mask, enrollments.final_grade.ge(60), enrollments.final_grade.lt(60)], ['In Progress', 'Passed', 'Failed'], default = 'Unknown')



In [12]:
# Create credits attempted and earned column

enrollments['credits_earned'] = np.where(enrollments.course_outcome.eq('Passed'), enrollments.credits_attempted, 0.0)

enrollments['attempted_number'] = 1

In [13]:
enrollments.drop(columns = ['year_start'], inplace = True)

display(enrollments.head())

,enrollment_id,student_id,course_id,school_year,credits_attempted,enrollment_status,final_grade,course_outcome,credits_earned,attempted_number
0,ENR00001,1145269,ELEC00013,2026-27,1.0,In Progress,NaN,In Progress,0.0,1
1,ENR00002,1145269,SCI00002,2026-27,1.0,In Progress,NaN,In Progress,0.0,1
2,ENR00003,1145269,MATH00006,2026-27,1.0,In Progress,NaN,In Progress,0.0,1
3,ENR00004,1145269,ART00008,2026-27,1.0,In Progress,NaN,In Progress,0.0,1
4,ENR00005,1145269,ART00002,2026-27,1.0,In Progress,NaN,In Progress,0.0,1


In [14]:
# Validate that dataset was created correctly

assert enrollments.enrollment_id.is_unique

assert enrollments.student_id.isin(students.student_id).all()

assert enrollments.course_id.isin(courses.course_id).all()

assert (enrollments.credits_earned <= enrollments.credits_attempted).all()

assert enrollments.loc[~enrollments.course_outcome.eq('Passed'), 'credits_earned'].eq(0).all()

In [15]:
folder_path = Path('/content/drive/MyDrive/Graduation_Tracker/data/ground_truth')

enrollments_file = folder_path / 'course_enrollments.csv'

enrollments.to_csv(enrollments_file, index = False)